[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/VaR_CEE_FM/blob/main/VaR_CEE_Figures/VaR_CEE_Figures.ipynb)

# VaR_CEE_Figures

Generates all six publication-quality figures for the paper:
1. VaR breach plots (GJR-GARCH vs TimesFM-2.5-Conf)
2. Violation rate comparison across models
3. Basel traffic light dashboard heatmap
4. ES adequacy (Acerbi-Szekely Z2) bar chart
5. Kupiec p-value heatmap
6. Cross-country model ranking

All figures use transparent backgrounds and legends positioned outside the plot area.

**Published in:** *Can Foundation Models Manage Risk? Zero-Shot VaR and ES Forecasting for CEE Markets*

**Author:** Daniel Traian Pele

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from pathlib import Path

%matplotlib inline

# ── Configuration ──────────────────────────────────────────
MARKETS = {
    "Romania":  {"index": "BET",   "fx": "EURRON"},
    "Poland":   {"index": "WIG20", "fx": "EURPLN"},
    "Czechia":  {"index": "PX",    "fx": "EURCZK"},
    "Hungary":  {"index": "BUX",   "fx": "EURHUF"},
    "Bulgaria": {"index": "SOFIX", "fx": "USDBGN"},
}
VAR_ALPHAS = [0.01, 0.025, 0.05]
BASELINE_MODELS = ["HS", "GJR-GARCH", "ARIMA-Conformal", "LSTM-Conformal"]
FM_MODELS = ["Chronos-2", "TimesFM-2.5", "Moirai-2.0"]
FM_CONF_MODELS = ["Chronos-2-Conf", "TimesFM-2.5-Conf", "Moirai-2.0-Conf"]
ALL_MODELS = BASELINE_MODELS + FM_MODELS + FM_CONF_MODELS

def get_all_series():
    series = []
    for country, info in MARKETS.items():
        series.append((country, info["index"], f"{info['index']}_ret", "index"))
        series.append((country, info["fx"], f"{info['fx']}_ret", "fx"))
    return series

VAR_DIR = Path("var_forecasts")
BT_DIR = Path("backtesting")
VAR_DIR.mkdir(exist_ok=True)
BT_DIR.mkdir(exist_ok=True)

# Publication style
plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'savefig.transparent': True,
})

## Upload Data Files

Upload two sets of files:
1. **VaR forecast CSVs** (all model outputs) \u2014 goes to `var_forecasts/`
2. **Backtesting results** (`backtesting_results.csv`) \u2014 goes to `backtesting/`

In [ ]:
from google.colab import files
print("Upload VaR forecast CSV files:")
uploaded_var = files.upload()
for fname, content in uploaded_var.items():
    dest = VAR_DIR / fname
    with open(dest, "wb") as f:
        f.write(content)
print(f"Uploaded {len(uploaded_var)} VaR files")

In [ ]:
print("\nUpload backtesting_results.csv:")
uploaded_bt = files.upload()
for fname, content in uploaded_bt.items():
    dest = BT_DIR / fname
    with open(dest, "wb") as f:
        f.write(content)
print(f"Uploaded {len(uploaded_bt)} backtesting files")

In [ ]:
def get_model_color(m):
    if m in FM_CONF_MODELS:
        return '#2ecc71'
    elif m in FM_MODELS:
        return 'coral'
    else:
        return 'steelblue'

def load_var_data(model, series_id):
    fname = VAR_DIR / f"{model}_{series_id}.csv"
    if not fname.exists():
        return None
    return pd.read_csv(fname, parse_dates=["date"])

def load_backtesting():
    bt_path = BT_DIR / "backtesting_results.csv"
    if not bt_path.exists():
        return None
    return pd.read_csv(bt_path)

In [ ]:
# ============================================================
# FIGURE 1: VaR BREACH PLOT
# ============================================================
print("Figure 1: VaR breach plots...")
all_series = get_all_series()
index_series = [(c, ri, s, t) for c, ri, s, t in all_series if t == "index"]

fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=False)
axes = axes.flatten()

garch_model = "GJR-GARCH"
best_fm = "TimesFM-2.5-Conf"

for i, (country, raw_id, series_id, _) in enumerate(index_series):
    if i >= 5:
        break
    ax = axes[i]
    df_garch = load_var_data(garch_model, series_id)
    df_fm = load_var_data(best_fm, series_id)
    plotted = False

    if df_garch is not None and len(df_garch) > 0:
        var_col = [c for c in df_garch.columns if "var_01" in c or "var_0.01" in c]
        if not var_col:
            var_col = [c for c in df_garch.columns if "var_" in c]
        if var_col:
            var_col = var_col[0]
            ax.plot(df_garch["date"], df_garch["y_true"],
                    color="black", linewidth=0.3, alpha=0.7, label="Returns")
            ax.plot(df_garch["date"], df_garch[var_col],
                    color="gray", linewidth=0.5, linestyle="--",
                    label="GARCH VaR 1%", alpha=0.8)
            breaches = df_garch["y_true"] < df_garch[var_col]
            if breaches.any():
                ax.scatter(df_garch.loc[breaches, "date"],
                           df_garch.loc[breaches, "y_true"],
                           color="red", s=8, zorder=5, alpha=0.6, label="Breach")
            plotted = True

    if df_fm is not None and len(df_fm) > 0:
        var_col = [c for c in df_fm.columns if "var_01" in c or "var_0.01" in c]
        if not var_col:
            var_col = [c for c in df_fm.columns if "var_" in c]
        if var_col:
            var_col = var_col[0]
            if not plotted:
                ax.plot(df_fm["date"], df_fm["y_true"],
                        color="black", linewidth=0.3, alpha=0.7, label="Returns")
            ax.plot(df_fm["date"], df_fm[var_col],
                    color="blue", linewidth=0.5,
                    label=f"{best_fm} VaR 1%", alpha=0.8)

    ax.set_title(f"{country} ({series_id.replace('_ret', '')})")
    ax.set_ylabel("Log returns")
    ax.axhline(y=0, color="gray", linewidth=0.3)

if len(index_series) < 6:
    axes[5].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.02), frameon=False, fontsize=9)
fig.suptitle("VaR 1% Breach Analysis: GJR-GARCH vs TimesFM-2.5-Conf", fontsize=12)
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
fig.savefig("fig1_var_breach.png", transparent=True, bbox_inches='tight')
fig.savefig("fig1_var_breach.pdf", transparent=True, bbox_inches='tight')
plt.show()
print("  Saved fig1_var_breach")

In [ ]:
# ============================================================
# FIGURE 2: VIOLATION RATE COMPARISON
# ============================================================
print("Figure 2: Violation rate comparison...")
df_bt = load_backtesting()

if df_bt is not None:
    alpha = 0.01
    df_a = df_bt[df_bt['alpha'] == alpha].copy()
    if len(df_a) == 0:
        alpha = 0.05
        df_a = df_bt[df_bt['alpha'] == alpha].copy()

    if len(df_a) > 0:
        fig, ax = plt.subplots(figsize=(14, 6))
        models = df_a['model'].unique()
        x = np.arange(len(models))
        avg_rates = [df_a[df_a['model'] == m]['violation_rate'].mean() for m in models]
        colors = [get_model_color(m) for m in models]
        bars = ax.bar(x, avg_rates, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
        ax.axhline(y=alpha, color='red', linestyle='--', linewidth=1, label=f'Expected ({alpha*100}%)')
        ax.set_xticks(x)
        ax.set_xticklabels(models, rotation=60, ha='right')
        ax.set_ylabel('Average violation rate')
        ax.set_title(f'VaR {alpha*100:.0f}% Violation Rates Across Countries')

        legend_elements = [
            Patch(facecolor='steelblue', edgecolor='black', label='Baselines'),
            Patch(facecolor='coral', edgecolor='black', label='Foundation Models'),
            Patch(facecolor='#2ecc71', edgecolor='black', label='Conformal FM'),
            plt.Line2D([0], [0], color='red', linestyle='--', label=f'Expected ({alpha*100}%)'),
        ]
        ax.legend(handles=legend_elements, loc='lower center',
                  bbox_to_anchor=(0.5, -0.30), ncol=4, frameon=False, fontsize=9)
        plt.tight_layout(rect=[0, 0.15, 1, 1])
        fig.savefig("fig2_violation_rates.png", transparent=True, bbox_inches='tight')
        fig.savefig("fig2_violation_rates.pdf", transparent=True, bbox_inches='tight')
        plt.show()
        print("  Saved fig2_violation_rates")
else:
    print("  No backtesting data available")

In [ ]:
# ============================================================
# FIGURE 3: TRAFFIC LIGHT DASHBOARD
# ============================================================
print("Figure 3: Traffic light dashboard...")
df_bt = load_backtesting()

if df_bt is not None:
    df_1 = df_bt[df_bt['alpha'] == 0.01].copy()
    if len(df_1) == 0:
        df_1 = df_bt[df_bt['alpha'] == df_bt['alpha'].min()].copy()

    if len(df_1) > 0:
        tl_map = {"Green": 0, "Yellow": 1, "Red": 2}
        df_1["tl_num"] = df_1["basel_tl"].map(tl_map).fillna(1)
        pivot = df_1.pivot_table(index='model', columns='series',
                                  values='tl_num', aggfunc='first')

        if not pivot.empty:
            fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns) * 0.8),
                                             max(4, len(pivot.index) * 0.5 + 1)))
            cmap = mcolors.ListedColormap(['#2ecc71', '#f39c12', '#e74c3c'])
            bounds = [-0.5, 0.5, 1.5, 2.5]
            norm = mcolors.BoundaryNorm(bounds, cmap.N)
            im = ax.imshow(pivot.values, cmap=cmap, norm=norm, aspect='auto')

            ax.set_xticks(range(len(pivot.columns)))
            ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=7)
            ax.set_yticks(range(len(pivot.index)))
            ax.set_yticklabels(pivot.index, fontsize=8)

            for i in range(len(pivot.index)):
                for j in range(len(pivot.columns)):
                    val = pivot.values[i, j]
                    label = {0: "G", 1: "Y", 2: "R"}.get(val, "?")
                    ax.text(j, i, label, ha="center", va="center",
                            fontsize=7, fontweight="bold", color="white")

            ax.set_title("Basel Traffic Light Dashboard (VaR 1%)")
            legend_elements = [
                Patch(facecolor='#2ecc71', label='Green (accepted)'),
                Patch(facecolor='#f39c12', label='Yellow (questioned)'),
                Patch(facecolor='#e74c3c', label='Red (rejected)'),
            ]
            ax.legend(handles=legend_elements, loc='lower center',
                      bbox_to_anchor=(0.5, -0.22), ncol=3, frameon=False, fontsize=9)
            plt.tight_layout(rect=[0, 0.06, 1, 1])
            fig.savefig("fig3_traffic_light.png", transparent=True, bbox_inches='tight')
            fig.savefig("fig3_traffic_light.pdf", transparent=True, bbox_inches='tight')
            plt.show()
            print("  Saved fig3_traffic_light")
else:
    print("  No backtesting data available")

In [ ]:
# ============================================================
# FIGURE 4: ES ADEQUACY
# ============================================================
print("Figure 4: ES adequacy...")
df_bt = load_backtesting()

if df_bt is not None and 'as_z2' in df_bt.columns:
    df_es = df_bt.dropna(subset=['as_z2']).copy()
    if len(df_es) > 0:
        fig, ax = plt.subplots(figsize=(10, 5))
        models = df_es['model'].unique()
        x = np.arange(len(models))
        avg_z2 = [df_es[df_es['model'] == m]['as_z2'].mean() for m in models]
        colors = ['steelblue' if m in BASELINE_MODELS else 'coral' for m in models]
        ax.bar(x, avg_z2, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
        ax.axhline(y=0, color='red', linestyle='--', linewidth=1, label='H0: Z2 = 0')
        ax.set_xticks(x)
        ax.set_xticklabels(models, rotation=45, ha='right')
        ax.set_ylabel('Acerbi-Szekely Z2')
        ax.set_title('Expected Shortfall Adequacy (Z2 Test)')

        legend_elements = [
            Patch(facecolor='steelblue', edgecolor='black', label='Baselines'),
            Patch(facecolor='coral', edgecolor='black', label='Foundation Models'),
            plt.Line2D([0], [0], color='red', linestyle='--', label='H0: Z2 = 0'),
        ]
        ax.legend(handles=legend_elements, loc='lower center',
                  bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False, fontsize=9)
        plt.tight_layout(rect=[0, 0.08, 1, 1])
        fig.savefig("fig4_es_adequacy.png", transparent=True, bbox_inches='tight')
        fig.savefig("fig4_es_adequacy.pdf", transparent=True, bbox_inches='tight')
        plt.show()
        print("  Saved fig4_es_adequacy")
    else:
        print("  No ES results available")
else:
    print("  No backtesting data or no ES results")

In [ ]:
# ============================================================
# FIGURE 5: KUPIEC P-VALUE HEATMAP
# ============================================================
print("Figure 5: Kupiec p-value heatmap...")
df_bt = load_backtesting()

if df_bt is not None:
    for alpha in VAR_ALPHAS:
        df_a = df_bt[df_bt['alpha'] == alpha].copy()
        if len(df_a) == 0:
            continue

        pivot = df_a.pivot_table(index='model', columns='series',
                                  values='kupiec_p', aggfunc='first')
        if pivot.empty:
            continue

        fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns) * 0.8),
                                         max(4, len(pivot.index) * 0.5 + 1)))
        cmap = plt.cm.RdYlGn
        im = ax.imshow(pivot.values, cmap=cmap, vmin=0, vmax=1, aspect='auto')

        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index, fontsize=8)

        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                            fontsize=6, color="black" if val > 0.3 else "white")

        cbar = plt.colorbar(im, ax=ax, label="p-value", shrink=0.8)
        alpha_pct = str(alpha).replace("0.", "")
        ax.set_title(f"Kupiec Test p-values (VaR {alpha*100}%)")

        legend_elements = [
            Patch(facecolor='#2ecc71', label='Pass (p > 0.05)'),
            Patch(facecolor='#e74c3c', label='Fail (p < 0.05)'),
        ]
        ax.legend(handles=legend_elements, loc='lower center',
                  bbox_to_anchor=(0.4, -0.22), ncol=2, frameon=False, fontsize=9)
        plt.tight_layout(rect=[0, 0.06, 1, 1])
        fig.savefig(f"fig5_kupiec_heatmap_{alpha_pct}.png", transparent=True, bbox_inches='tight')
        fig.savefig(f"fig5_kupiec_heatmap_{alpha_pct}.pdf", transparent=True, bbox_inches='tight')
        plt.show()

    print("  Saved fig5_kupiec_heatmap")
else:
    print("  No backtesting data available")

In [ ]:
# ============================================================
# FIGURE 6: CROSS-COUNTRY MODEL RANKING
# ============================================================
print("Figure 6: Cross-country model ranking...")
df_bt = load_backtesting()

if df_bt is not None:
    fig, ax = plt.subplots(figsize=(8, 5))
    green_rates = df_bt.groupby('model')['basel_tl'].apply(
        lambda x: (x == 'Green').mean()
    ).sort_values(ascending=True)

    colors = ['coral' if m in FM_MODELS else 'steelblue' for m in green_rates.index]
    ax.barh(range(len(green_rates)), green_rates.values,
            color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(green_rates)))
    ax.set_yticklabels(green_rates.index)
    ax.set_xlabel('Green Zone Rate')
    ax.set_title('Cross-Country Model Ranking (Basel Green Zone Rate)')
    ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.5)

    legend_elements = [
        Patch(facecolor='steelblue', edgecolor='black', label='Baselines'),
        Patch(facecolor='coral', edgecolor='black', label='Foundation Models'),
    ]
    ax.legend(handles=legend_elements, loc='lower center',
              bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False, fontsize=9)
    plt.tight_layout(rect=[0, 0.06, 1, 1])
    fig.savefig("fig6_model_ranking.png", transparent=True, bbox_inches='tight')
    fig.savefig("fig6_model_ranking.pdf", transparent=True, bbox_inches='tight')
    plt.show()
    print("  Saved fig6_model_ranking")
else:
    print("  No backtesting data available")

In [ ]:
print("\n=== All figures generated ===")